# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Optimisation as Architecture: LISTA, ADMM-Net, CRATE & Mesa-Optimizers
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Gregor, K., & LeCun, Y. (2010). *Learning Fast Approximations of Sparse Coding*. ICML.
- Yang, Y., Sun, J., Li, H., & Xu, Z. (2018). *ADMM-Net: A Deep Architecture for Compressive Sensing MRI*. IEEE TPAMI.
- Yu, Y., et al. (2023). *White-Box Transformers via Sparse Coding and Rate Reduction (CRATE)*. NeurIPS.
- Von Oswald, J., et al. (2023). *Transformers learn in-context by gradient descent*. ICML.
- Hubinger, E., et al. (2019). *Risks from Learned Optimization in Advanced Machine Learning Systems*. arXiv:1906.01820.

## 2. Mathematical Foundations & Governing Equations

### LISTA Unrolled Formulation
Unrolling the Iterative Shrinkage-Thresholding Algorithm (ISTA) yields:
$$z_{k+1} = \eta_{\theta_k}(z_k W_s + y W_e),$$
where $\eta_\theta(x) = \operatorname{sign}(x) \odot \max(|x| - \theta, 0)$ is the soft-thresholding operator.

### ADMM-Net Step
Solves composite problem $\min_{x, z} \frac{1}{2}\|A x - y\|_2^2 + \lambda \|z\|_1$ s.t. $x - z = 0$:
1. $x_{k+1} = (y + \rho(z_k - u_k)) W_{\text{inv}}$
2. $z_{k+1} = \eta_{\theta/\rho}(x_{k+1} + u_k)$
3. $u_{k+1} = u_k + x_{k+1} - z_{k+1}$

### CRATE Coding Rate Reduction Step
Optimizes sparse representation via dictionary $D$:
$$z_{k+1} = \eta_\lambda\bigl( z_k - \eta z_k (D^T D - I) \bigr).$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import numpy as np

torch.manual_seed(42)
B, M, N = 4, 3, 5

# LISTA parameters
y_meas = torch.randn(B, M) * 0.8
w_e = torch.randn(M, N) * 0.4
raw_ws = torch.randn(N, N) * 0.2
w_s = raw_ws / (torch.linalg.matrix_norm(raw_ws, ord=2) * 1.5)
theta_lista = torch.abs(torch.randn(N)) * 0.05 + 0.02

# ADMM-Net parameters
y_admm = torch.randn(B, N) * 0.6
raw_winv = torch.randn(N, N) * 0.3
w_inv = raw_winv.t() @ raw_winv + 0.5 * torch.eye(N)

# CRATE parameters
d_dict = torch.randn(M, N) * 0.5
z_crate_in = torch.randn(B, N) * 0.5

print(f"Batch B: {B}, Measurement M: {M}, Sparse latent N: {N}")
print(f"LISTA lateral matrix norm: {torch.linalg.matrix_norm(w_s, ord=2):.4f} (< 1.0, contractive)")

In [ ]:
# 4. First-Principles Implementation & Native Parity
def soft_threshold(x, th):
    return torch.sign(x) * torch.clamp(torch.abs(x) - th, min=0.0)

# 1. LISTA 3-layer unroll
z_lista = torch.zeros(B, N)
for _ in range(3):
    lin = z_lista @ w_s + y_meas @ w_e
    z_lista = soft_threshold(lin, theta_lista)

# 2. ADMM-Net 1-step unroll
x0 = torch.zeros(B, N)
z0 = torch.zeros(B, N)
u0 = torch.zeros(B, N)
rho = 1.0
theta_admm = 0.08
rhs_admm = y_admm + rho * (z0 - u0)
x1 = rhs_admm @ w_inv
z1 = soft_threshold(x1 + u0, torch.tensor(theta_admm / rho))
u1 = u0 + x1 - z1

# 3. CRATE proximal step
dtd = d_dict.t() @ d_dict
gram_eye = dtd - torch.eye(N)
grad_crate = z_crate_in @ gram_eye
z_crate_out = soft_threshold(z_crate_in - 0.1 * grad_crate, torch.tensor(0.04))

print(f"LISTA latent norm: {torch.norm(z_lista):.4f}, ADMM-Net x1 norm: {torch.norm(x1):.4f}")
print(f"CRATE output norm: {torch.norm(z_crate_out):.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Sparsity promotion via soft-thresholding
zero_count_lista = (z_lista == 0.0).sum().item()
total_elements = z_lista.numel()
print(f"LISTA sparsity ratio: {zero_count_lista} / {total_elements} zeroes ({(zero_count_lista/total_elements)*100:.1f}%)")
assert zero_count_lista > 0, "Soft-thresholding failed to promote sparsity"

# Invariant 2: ADMM dual multiplier residual consistency
diff_primal = x1 - z1
assert torch.norm(u1 - (u0 + diff_primal)).item() < 1e-6, "ADMM dual multiplier update inconsistent"

# Invariant 3: CRATE output sparsity
zero_count_crate = (z_crate_out == 0.0).sum().item()
assert zero_count_crate > 0, "CRATE soft-thresholding failed to promote sparsity"

print(f"All structural and mathematical invariants verified successfully.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 200
t0 = time.perf_counter()
for _ in range(iters):
    _ = z_lista @ w_s + y_meas @ w_e
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"LISTA layer evaluation latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Negative threshold in soft-thresholding violates proximal contraction
th_bad = torch.tensor([-0.5])
x_test = torch.tensor([[-0.2, 0.2]])
st_bad = soft_threshold(x_test, th_bad)
print(f"Pathological negative threshold expands amplitude: {st_bad.tolist()}")

# Corrected: Enforcing strictly non-negative threshold guarantees non-expansiveness
th_corr = torch.clamp(th_bad, min=0.0)
st_corr = soft_threshold(x_test, th_corr)
print(f"Corrected clamped threshold output: {st_corr.tolist()}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Inductive Bias via Unrolling**: Embedding mathematical optimization algorithms as feedforward layers yields interpretable networks with guaranteed structural properties.
2. **Sparsity from Proximal Layers**: Soft-thresholding acts as a non-linear proximal activation that naturally produces sparse representations.
3. **Transformers as Unrolled Solvers**: CRATE demonstrates that canonical Transformer attention and MLP blocks correspond to unrolled coding rate reduction algorithms.